In [38]:
import cv2
from python_orb_slam3 import ORBExtractor
import numpy as np
import math

In [27]:
def orb_bits(hexstr):
  b = bytes.fromhex(hexstr)                       # 32 bytes
  return [(b[i // 8] >> (i % 8)) & 1 for i in range(256)]   # bits[i] = test i

def hamming(h1, h2):
  return bin(int(h1, 16) ^ int(h2, 16)).count("1")

In [3]:
def load_rectification(settings_path):
    """Reproduce ORB-SLAM3 Settings::precomputeRectificationMaps for cam0."""
    fs = cv2.FileStorage(settings_path, cv2.FILE_STORAGE_READ)
    if not fs.isOpened():
        sys.exit(f"Cannot open settings file {settings_path}")
    g = lambda k: fs.getNode(k).real()

    def cam(i):
        K = np.array([[g(f"Camera{i}.fx"), 0, g(f"Camera{i}.cx")],
                      [0, g(f"Camera{i}.fy"), g(f"Camera{i}.cy")],
                      [0, 0, 1]])
        D = np.array([g(f"Camera{i}.k1"), g(f"Camera{i}.k2"), g(f"Camera{i}.p1"), g(f"Camera{i}.p2")])
        return K, D

    K1, D1 = cam(1)
    K2, D2 = cam(2)
    size = (int(g("Camera.width")), int(g("Camera.height")))
    T_c1_c2 = fs.getNode("Stereo.T_c1_c2").mat().astype(np.float64)
    fs.release()

    T_c2_c1 = np.linalg.inv(T_c1_c2)  # ORB-SLAM3 passes Tlr.inverse()
    R12 = np.ascontiguousarray(T_c2_c1[:3, :3])
    t12 = np.ascontiguousarray(T_c2_c1[:3, 3]).reshape(3, 1)
    R1, R2, P1, P2, Q, _, _ = cv2.stereoRectify(K1, D1, K2, D2, size, R12, t12,
                                                flags=cv2.CALIB_ZERO_DISPARITY, alpha=-1, newImageSize=size)
    m1, m2 = cv2.initUndistortRectifyMap(K1, D1, R1, P1[:3, :3], size, cv2.CV_32FC1)
    return m1, m2, P1[:3, :3], size

In [95]:
def load_landmarks(path):
    """Returns ids (N,), xyz (N,3), rep_desc (N,32) uint8, all_desc {id: (M,32) uint8}."""
    landmark_dict = {}
    ids, xyz, orb_desc, orb_hex = [],[], [], []
    
    with open(path) as f:
        for line in f:
            t = line.split()
            temp = {}
            if(t[0] == '#'):
                continue
            ids.append(int(t[0]))
            xyz.append( ([float(t[1]), float(t[2]), float(t[3])]) )
            orb_desc.append(np.frombuffer(bytes.fromhex(t[4]), np.uint8))
            orb_hex.append(t[4])
    
    landmark_dict['ids'] = np.array(ids)
    landmark_dict['xyz'] = np.array(xyz)
    landmark_dict['orb_desc'] = np.stack(orb_desc)
    landmark_dict['orb_hex'] = np.stack(orb_hex)

    return landmark_dict

In [104]:
def print_landmark(i):
    print(f"Id:{all_landmarks['ids'][i]}, xyz: {all_landmarks['xyz'][i]}, orb_hex: {all_landmarks['orb_hex'][i]}")

In [209]:
# test time: 1403636628763555584.png

imgpath = "/home/shiladitya/Dataset/euroc/MH_01_easy/mav0/cam0/data/1403636639263555584.png"
setting_path = "EuRoC.yaml"
nfeatures = 1200

In [210]:
print_landmark(27)

Id:101, xyz: [ 2.12057257 -0.93725151  5.05071831], orb_hex: 5ead26889d724beae6158b02cee23a77e9c67ab13e0bafddb9a08b5f8047b766


In [211]:
## Get all landmarks in first camera keyframe coords 
all_landmarks = load_landmarks("landmarks_MH01_stereo_points.txt")

In [212]:
raw = cv2.imread(imgpath)


In [213]:
m1, m2, K, (W, H) = load_rectification(setting_path)

In [214]:
img = cv2.remap(raw, m1, m2, cv2.INTER_LINEAR)

In [215]:
cv2.imshow('rectified_img', img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [216]:
fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]

In [217]:
# 2. ORB features (parameters from EuRoC.yaml / ORBextractor.cc)
orb = cv2.ORB_create(nfeatures=nfeatures, scaleFactor=1.2, nlevels=8, edgeThreshold=19,
                     patchSize=31, fastThreshold=20, scoreType=cv2.ORB_FAST_SCORE)
kps, desc = orb.detectAndCompute(img, None)

In [218]:
if desc is None:
    sys.exit("No ORB features detected")
kp_uv = np.array([k.pt for k in kps])

In [219]:
kp_uv

array([[656.        , 197.        ],
       [377.        , 306.        ],
       [545.        , 458.        ],
       ...,
       [372.65090942, 361.90136719],
       [634.22320557, 236.49000549],
       [480.1463623 , 128.99455261]])

In [220]:
all_landmarks['orb_hex'][2345]

'0f37be7bb97f3ffbf5eedfb5dbfdffefdf7eff3ff5f1fd8a7e7f25c7ceffacd7'

In [237]:
candidate_desc = "39b15347524f62786561eb6c531388b4369c466a19b27d0d37ac20046b83e2b0" #"5547ad672c0e4f3964ea9f4cd321996b14c47c6ee9f0fd08e4afe5946cfb5071" #"6c1bfc77ad2f5739efbebf08df7b411a8d766d0a1860eb20cfff6135c2eb4623"
raw_bytes = bytes.fromhex(candidate_desc)
candidate_desc_dist = np.frombuffer(raw_bytes, np.uint8)
print(candidate_desc_dist)

[ 57 177  83  71  82  79  98 120 101  97 235 108  83  19 136 180  54 156
  70 106  25 178 125  13  55 172  32   4 107 131 226 176]


In [238]:
min_dist = math.inf
lowest = 0;
for i, d1 in enumerate(desc): #= "6c1bfc77ad2f5739efbebf08df7b411a8d766d0a1860eb20cfff6135c2eb4623"
    # raw_bytes = bytes.fromhex(hex_string)
    # d1 = np.frombuffer(raw_bytes, np.uint8)
    dist = cv2.norm(d1, candidate_desc_dist, cv2.NORM_HAMMING)

    if(dist < min_dist):
        lowest = i
        min_dist = dist
        

In [239]:
min_dist,lowest

(51.0, 148)

In [240]:
min_dist,lowest

(51.0, 148)

In [241]:
desc[lowest]

array([ 45, 113,  89, 101,  86,  79,  99, 120, 124, 224, 107,  72, 215,
        21, 137,  48,  50,  28,  22, 106,  65, 176, 113,  14, 229, 237,
        33,   4, 104, 163,  67, 177], dtype=uint8)

In [242]:
kp_uv[lowest]

array([632., 292.])

In [243]:
def get_3Dlandmarks_in_curr_frame(landmark_map, curr_frame_orb_desc):
    # landmark_map is actually all_landmarks

    landmarks_seen = []

    for curr_desc in curr_frame_orb_desc:
        min_dist = math.inf
        lowest = 0
        for i, d1 in enumerate(landmark_map['orb_desc']): #= "6c1bfc77ad2f5739efbebf08df7b411a8d766d0a1860eb20cfff6135c2eb4623"
            # raw_bytes = bytes.fromhex(hex_string)
            # d1 = np.frombuffer(raw_bytes, np.uint8)
            dist = cv2.norm(d1, curr_desc, cv2.NORM_HAMMING)
        
            if(dist < min_dist):
                lowest = i
                min_dist = dist
        landmarks_seen.append([lowest,landmark_map['ids'][lowest] , min_dist])

    return landmarks_seen


In [244]:
landmarks_seen = get_3Dlandmarks_in_curr_frame(all_landmarks, desc)

In [245]:
len(all_landmarks['orb_hex'])

7952

In [246]:
len(landmarks_seen)

1230

In [249]:
print_landmark(landmarks_seen[0][0])

Id:44281, xyz: [ 9.66007805 -0.69196951 12.13240528], orb_hex: a567404bdd39c6ee5122a918c3681f9dd9ca422a7e26cc3e269c053d537bda84


In [254]:
landmarks_seen

[[7950, 44281, 57.0],
 [2313, 18839, 54.0],
 [6425, 15793, 52.0],
 [369, 2000, 42.0],
 [5499, 40724, 53.0],
 [2405, 18071, 46.0],
 [3947, 29718, 53.0],
 [6783, 17761, 58.0],
 [903, 6255, 48.0],
 [2343, 18359, 30.0],
 [6383, 18494, 47.0],
 [1538, 13270, 46.0],
 [6392, 28852, 48.0],
 [7880, 37789, 61.0],
 [6704, 18926, 28.0],
 [7122, 30527, 50.0],
 [4105, 31159, 45.0],
 [4886, 36701, 51.0],
 [4105, 31159, 47.0],
 [2367, 18465, 6.0],
 [7234, 12237, 35.0],
 [7463, 23674, 56.0],
 [6624, 29563, 49.0],
 [4652, 34344, 30.0],
 [4350, 33326, 48.0],
 [6505, 32815, 31.0],
 [6010, 46724, 36.0],
 [5755, 43103, 50.0],
 [3512, 26318, 30.0],
 [1695, 13748, 48.0],
 [7008, 18144, 23.0],
 [7613, 48203, 43.0],
 [448, 2763, 53.0],
 [3909, 31029, 39.0],
 [2389, 18040, 31.0],
 [204, 1674, 52.0],
 [4831, 37116, 35.0],
 [2385, 18024, 20.0],
 [7199, 18480, 39.0],
 [7322, 18539, 23.0],
 [3835, 30718, 65.0],
 [1477, 12762, 35.0],
 [2421, 18300, 31.0],
 [1975, 16349, 60.0],
 [3886, 30880, 40.0],
 [420, 2262, 45.0],

In [252]:
landmarks_seen_sorted = sorted(landmarks_seen, key=lambda x: x[2])

In [253]:
landmarks_seen_sorted

[[2291, 17692, 0.0],
 [2362, 18451, 1.0],
 [2239, 18615, 2.0],
 [2275, 18779, 3.0],
 [6731, 18907, 5.0],
 [2367, 18465, 6.0],
 [7192, 18498, 6.0],
 [7161, 18512, 6.0],
 [2357, 18425, 6.0],
 [2351, 18406, 7.0],
 [2333, 18286, 7.0],
 [2341, 18354, 8.0],
 [2423, 18323, 8.0],
 [6733, 18911, 9.0],
 [2365, 18458, 9.0],
 [7006, 18124, 9.0],
 [7317, 18528, 10.0],
 [6383, 18494, 10.0],
 [2421, 18300, 10.0],
 [7759, 18106, 10.0],
 [2386, 18028, 10.0],
 [2363, 18452, 10.0],
 [2290, 17691, 10.0],
 [6380, 18489, 11.0],
 [2343, 18359, 11.0],
 [2243, 18634, 11.0],
 [6787, 17767, 11.0],
 [7312, 18885, 12.0],
 [2371, 18473, 12.0],
 [2030, 18260, 12.0],
 [2404, 18070, 12.0],
 [7759, 18106, 12.0],
 [7315, 18883, 14.0],
 [2444, 19682, 14.0],
 [7313, 18889, 14.0],
 [7317, 18528, 14.0],
 [7323, 18540, 14.0],
 [2047, 17009, 15.0],
 [2418, 18099, 15.0],
 [7008, 18144, 16.0],
 [6786, 17765, 16.0],
 [6703, 18923, 17.0],
 [2386, 18028, 17.0],
 [6786, 17765, 17.0],
 [2047, 17009, 17.0],
 [7319, 18532, 18.0],
 [23

In [208]:
print_landmark(2362)

Id:18451, xyz: [ 6.5620327  -1.08174705 11.61051559], orb_hex: 5cbeb96332ef53b2366dbf2ddf37c2109717ed109b18f937cbcf31f5e8c30773
